# 09 · 投影与 SQLite 持久化

你给一个商务核查会话追加「顺便查一下证书有效期」，并让它排队。随后关闭客户端再打开。追到本地数据库，可以查明 **后台在关闭前已经完成了哪一步**。如果该输入已接收但未交付，数据库应能回答「还有一条待处理输入」。如果已经交付，历史应能回答「这条消息何时成为会话上下文的一部分」。两者复用同一个消息 ID，落在不同的表里。



## 1 · 先找到数据库，再区分各种状态

固定提交的 CLI 服务把 `databasePath(global.data)` 作为数据库路径传给 Server。支持常见发行通道的默认数据库名是 `opencode.db`。没有设置 `XDG_DATA_HOME` 时，常见路径为 `~/.local/share/opencode/opencode.db`。设置了 `XDG_DATA_HOME` 时，数据目录改为该根目录下的 `opencode`。自定义通道可使用不同文件名，`OPENCODE_DB` 可指定覆盖值。需要确认某次 CLI 启动将选什么路径时，可以看 `opencode debug paths db`。嵌入式 SDK 默认为 `:memory:`，并允许调用方覆盖。因此测试或嵌入式会话是否跨进程存在，要先看实际用的是文件库还是内存库。[CLI 服务传参](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/cli/src/server-process.ts#L101-L116) · [数据库路径选择](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/cli/src/database-path.ts#L4-L12) · [数据目录](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/util/src/global-roots.ts#L4-L17) · [SDK 默认选项](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/sdk/src/internal/host.ts#L34-L42)

本地 Bun 驱动支持调优。数据库层在支持的驱动上请求 WAL、`synchronous=NORMAL` 和 `busy_timeout=5000`，支持外键切换时启用外键，然后执行迁移。WAL 属于 **SQLite 的事务日志**，用于处理数据库写入。OpenCode 的 `event` 表则属于 **应用历史**，是否保留事件负载由 `events.persist` 控制。在这一提交中，Bus 未配置时默认关闭事件负载保留，仍分配聚合序号并更新投影。文件库上的当前状态可以跨进程保留，历史事件能否查询或重放要另查实例设置。[数据库初始化](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/database/database.ts#L35-L54) · [Bun 驱动](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/database/sqlite.bun.ts#L1-L14) · [Server 向 Bus 传参](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/server/src/routes.ts#L146-L148) · [Bus 默认选项](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/bus.ts#L204-L215)

| 要回答的问题 | 看哪张表 | 怎样识别 |
|---|---|---|
| 这个 Session 在哪里，当前选了谁？ | `session_v2` | `id`、`project_id`、`directory`、`workspace_id`、`parent_id`、`agent`、`model`。父子关系与工作位置都可在这里查到。[会话表](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/sql.ts#L28-L84) |
| 用户输入已接收，还在等交付吗？ | `session_inbox` | `id` 标识输入，`type`、`payload`、`delivery` 描述它的内容与投递方式，`enqueued_seq` 给出接收顺序。[Inbox 表](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/sql.ts#L130-L152) |
| 这轮模型输出和工具调用已经记录了什么？ | `session_message`、`session_part` | 消息行的 `seq` 用于会话内排序。助手内容可拆成按 `ordinal` 排序的 part，工具 part 的 `call_id` 对应那次工具调用。[消息与分段表](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/sql.ts#L86-L128) |
| 这组 durable 事件到了几号，能读到事件正文吗？ | `event_sequence`、`event` | 前者按 `aggregate_id` 保存最近的 `seq` 和可选的重放用 `owner_id`。开启负载保留时，后者保存事件 ID、序号、带版本的类型和负载。[事件表](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/event/sql.ts#L4-L26) |
| 上一次执行中断后，是否仍有待恢复的认领？ | `session_v2` | `time_suspended` 是执行 claim 的历史列名，`resume_attempts` 记录恢复次数。界面查询用的 `time_busy`、`time_idle`、`idle_outcome` 是另一组信息。[执行认领读写](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/store.ts#L245-L301) · [空闲投影](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L841-L863) |
| 后台检查工具在跑，还是已有结果等交给父会话？ | `job` | 用 `notification_id` 标识通知，结合 `kind`、`status`、`session_id`、`child_session_id`、`recovery`、`output`、`error` 辨认。待投递报告也用这张表。[Job 表](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/job/sql.ts#L7-L27) · [子任务结算](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/execution/subagent-report.ts#L151-L188) |
| 指令变化和较大的媒体内容怎么存？ | `instruction_state`、`instruction_entry`、`instruction_blob`，以及 `blob`、`blob_ref` | 指令快照包含 epoch 起点和已覆盖序号。开启媒体外置后，投影可保存引用，字节放在按哈希标识的 blob，读取时组装。[指令和媒体表](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/sql.ts#L154-L215) · [更新指令投影](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L1074-L1079) |

## 2 · 投影把「发生了什么」变成「现在该读什么」

Runner 和消息页面经常需要读取同一个会话的当前历史。如果每次都重算一遍历史事件，就要重新做排队、分段、用量累加和兼容转换。OpenCode 在 durable 事件提交时调用对应的投影器。投影器把变化写到应用平时直接查询的表里。例如 `Step.Ended` 同时更新消息状态和累计用量，`Execution.Started` 则更新会话的忙碌时间。Runner 需要请求上下文时，从投影读取历史与指令基线，按上下文边界选取可用内容。[Bus 提交顺序](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/bus.ts#L396-L432) · [会话生命周期投影](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L1065-L1119) · [Runner 历史读取](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/history.ts#L282-L307)

一次提交也能带上运行时自己的数据库写入。例如执行开始时，`Execution.Started` 的投影记录 `time_busy`，调用者的提交回调建立 claim。子 Agent 正常结算时，另一事务释放认领并记录待投递回报，提交后才向父会话投递。**投影回答当前可读状态，claim 和待投递行回答重启后还有什么义务。** 是否继续执行还要看 claim 和恢复策略。[执行开始及认领](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/execution.ts#L78-L85) · [执行生命周期连接](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/execution.ts#L138-L176) · [提交后回报](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/execution/subagent-report.ts#L179-L199)

## 3 · 同一个消息 ID 怎样从待办进入历史



1. **接收时。** `InboxEnqueued` 投影在 `session_inbox` 插入一行，`enqueued_seq` 为接收事件的序号。选择 `queue` 的输入可以待在这里，直到运行器抵达允许的交付边界。[接收投影](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L1040-L1050) · [首条排队输入选择](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/inbox.ts#L470-L499)
2. **交付时。** `InboxDelivered` 的投影先执行删除并返回待交付行，再用它的内容插入 `session_message`。用户与合成输入沿用原 ID，消息 `seq` 取本次**交付事件**序号。两个动作在同一事件事务内提交。若投影后续写入失败，整个事务回滚，待交付行也会回来。压缩、会话移动等控制项同样消耗待交付行，其各自的动作有自己的事件。[交付投影](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L1011-L1039) · [消费输入](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/inbox.ts#L347-L375) · [消息序号写入](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L801-L820) · [控制项交付](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/runner/llm.ts#L91-L117)
3. **读取时。** 历史按消息 `seq` 排序。同一个 ID 可以连接接收和交付，仍需用序号判断消息何时对历史查询可见。重放测试从保留的接收事件重建待交付行，断言消息历史为空，执行唤醒也为空。重新得到待办记录，并没有自动开始推进它。[历史排序](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/history.ts#L204-L213) · [待交付重投影测试](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-prompt.test.ts#L1156-L1211)

## 4 · 存储形状会演进，读取语义可以保持稳定

一个助手消息可以先有文字，再提出工具调用，最后填入对应结果。常规助手更新在 `experimental.part_rows` 未设为 false 时使用分段写入。`session_message` 保存消息主体与元数据，`session_part` 保存内容分段，工具结果可更新对应 part。旧的整条内联消息仍能读出来。如果一条消息既有 part 行，又有旧版写入的内联内容，读取端依次拼接分段与内联内容，然后恢复媒体引用。通过测试切换三次存储配置，最终读到的回复仍等价。[写入适配器选择](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/projector.ts#L316-L343) · [历史读时组装](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/src/session/history.ts#L90-L124) · [跨存储形状验证](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-projector.test.ts#L2581-L2612)

媒体也有同类边界。原版测试把同一张截图附上两次，又让工具返回相同字节。打开 `experimental.media_blob` 后，待交付投影、用户消息、工具 part 中都可看到 `media:<hash>`。`blob` 表只有一份该媒体字节。历史读取再把引用恢复为文件内容。这里的去重结论针对投影中的 blob 存储，历史事件负载保留是另一层配置。[媒体去重与恢复验证](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-projector.test.ts#L1470-L1562)

## 在 Notebook 里观察四件事

| 案例 | 观察和判断 |
|---|---|
| **队列中的补充检查** | 运行原版交付测试，按同一个 ID 查看两张表。交付后 Inbox 行消失，用户消息行的 `seq` 等于交付事件序号。业务对应场景是合规核查的补充说明从排队进入历史。[交付断言](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-projector.test.ts#L351-L375) |
| **分段保存的编码回复** | 比较全程启用 part、全程关闭 part 和中途切换三条路径。猜测历史 API 会不会因此返回三个不同结果，再检查上游断言。[分段兼容测试](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-projector.test.ts#L2581-L2612) |
| **两份截图和一个工具输出** | 看哪个表只有一份二进制，哪个表记录使用处。再确认模型读取时得到的是可用文件内容。[媒体引用测试](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-projector.test.ts#L1470-L1562) |
| **重放排队中的审批备注** | 原版测试依序重放保存的普通与合成输入事件。观察待交付行重新出现时，模型是否随之执行。审批备注是对测试的业务类比，测试输入为 `Replay pending` 和 `Replay synthetic`。[重放用例](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-prompt.test.ts#L1156-L1211) |

**自测。** 找到一个 Session 的用户消息行、一个仍为 running 的后台 Job，以及一个非空 `time_suspended`。分别说明它们证明了什么。下一章再看进程启动时，恢复程序如何为这些记录选择继续执行、补发报告或留在待交付状态。

### 沿同一个 ID 看实际的两次提交

用上游的录制问候实验，先以 `resume: false` 提交，再显式恢复。第一次采样时，`session_inbox` 有一行，`enqueued_seq=0`，消息表为空。运行结束时，inbox 清空，同一个 ID 已成为 `session_message` 中的用户消息，消息序号为 2。序号 1 属于中间提交的指令更新事件。助手消息序号为 3。这些数来自开启事件载荷留存的课堂用例，可以在 Notebook 里同时对照行状态与事件顺序。

## 实验 A · 核对已跑过的原版案例

读取随课保存的实跑计数、原版测试入口和对应断言。断言说明来自原版测试，计数来自逐案例执行。模型回复使用上游确定性夹具。

In [1]:
CHAPTER = '09-projections-storage'

from pathlib import Path
import json
from IPython.display import Markdown, display

here = Path.cwd().resolve()
course = next(path for path in (here, *here.parents) if (path / "fixtures/storage-cases.json").is_file())
catalog = json.loads((course / "fixtures/storage-cases.json").read_text())
cases = [case for case in catalog["cases"] if case["chapter"] == CHAPTER]
print("已验证 · OpenCode", catalog["commit"][:10], "/ Bun", catalog["bun"])
for case in cases:
    count = case["counts"]
    link = f'https://github.com/anomalyco/opencode/blob/{catalog["commit"]}/{case["source"]}#L{case["line"]}'
    display(Markdown(f'**{case["title"]}** · {count["pass"]} 通过、{count["fail"]} 失败 · [原版测试]({link})'))
    print("断言核对 ·", case["observation"])


已验证 · OpenCode ee5fe34e13 / Bun 1.4.4


**接收问候后显式恢复** · 1 通过、0 失败 · [原版测试](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-runner-recorded.test.ts#L185)

断言核对 · resume:false 先接收输入，显式 resume 后投影用户和助手消息，模型回复来自原版 HTTP 录制。


**交付同一条输入并保持顺序** · 2 通过、0 失败 · [原版测试](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-projector.test.ts#L351)

断言核对 · 输入投影消费待处理行，消息和历史按聚合序号排序。


**重放输入投影不启动 Runner** · 1 通过、0 失败 · [原版测试](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-prompt.test.ts#L1156)

断言核对 · 项目重放可重建待交付内容，重放本身不调度执行。


**助手子项和媒体引用** · 2 通过、0 失败 · [原版测试](https://github.com/anomalyco/opencode/blob/ee5fe34e136488a1839ac73bcca8321c51a31b75/packages/core/test/session-projector.test.ts#L1470)

断言核对 · 切换 part_rows 仍读取到同一条回复。开启 media_blob 后，多类会话对象通过引用使用同一份媒体内容。


## 实验 B · 在本机重跑原版流程

准备固定提交的独立源码副本与锁定依赖，启动 Jupyter 时设置 `LEARN_OPENCODE_SOURCE` 指向副本，然后运行下方单元格。它会执行同一组原版测试，核对通过数。排查入口见本书 README。

In [2]:
import os
import re
import shutil
import subprocess

source = os.environ.get("LEARN_OPENCODE_SOURCE")
if not source:
    print("已显示实跑记录。配置 LEARN_OPENCODE_SOURCE 可在本机重跑。")
else:
    repo = Path(source).expanduser().resolve()
    bun = shutil.which("bun")
    if bun is None:
        raise RuntimeError("请在 PATH 中提供 Bun " + catalog["bun"])
    actual = subprocess.check_output(
        ["git", "-C", str(repo), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual != catalog["commit"]:
        raise RuntimeError("原版源码副本的提交需与课程基准一致")
    version = subprocess.check_output([bun, "--version"], text=True).strip()
    if version != catalog["bun"]:
        raise RuntimeError("测试使用 Bun " + catalog["bun"] + "，当前版本为 " + version)
    environment = dict(os.environ)
    environment.pop("RECORD", None)
    environment["CI"] = "true"
    for case in cases:
        result = subprocess.run(
            [bun, *case["command"]],
            cwd=repo / "packages/core", env=environment,
            capture_output=True, text=True, timeout=45,
        )
        text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", result.stdout + result.stderr)
        counts = {key: int(number) for number, key in re.findall(r"(?m)^\s*(\d+) (pass|fail)\b", text)}
        expected = {key: case["counts"][key] for key in ("pass", "fail")}
        if result.returncode or counts != expected:
            raise RuntimeError(case["title"] + "\n" + text[-3000:])
        print(f'{case["title"]} · {counts["pass"]} pass · {counts["fail"]} fail')


接收问候后显式恢复 · 1 pass · 0 fail


交付同一条输入并保持顺序 · 2 pass · 0 fail


重放输入投影不启动 Runner · 1 pass · 0 fail


助手子项和媒体引用 · 2 pass · 0 fail


## 实验 C · 看实际 SQLite 行

下面对照同一运行的多个时点。数据来自在原版测试临时副本中增加的只读查询。恢复实验还把上游测试库切到隔离磁盘文件，在 Bun 退出后由 Python 重新打开。恢复扫描本身由上游测试宿主触发。

In [3]:
import os
import sys

# 默认读取已经验证的 SQLite 行。提供独立副本时，从实际执行重新采样。
source = os.environ.get("LEARN_OPENCODE_SOURCE")
if source:
    sys.dont_write_bytecode = True
    sys.path.insert(0, str(course / "fixtures"))
    from storage_probe import run_probe

    rows = run_probe(CHAPTER, source)
    print("已重跑原版流程并重新读取 SQLite 行")
else:
    captured = json.loads((course / "fixtures/storage-observations.json").read_text())
    assert captured["commit"] == catalog["commit"]
    rows = captured["observations"][CHAPTER]
    print("使用已保存的原版运行观测")

if CHAPTER == "08-events-and-bus":
    lines = ["| 场景 | 投影行数 | event 行数 | event_sequence |",
             "|---|---:|---:|---|"]
    for row in rows:
        seq = ", ".join(str(item["seq"]) for item in row["event_sequence"]) or "空"
        lines.append(f'| {row["phase"]} | {len(row["projection"])} | {len(row["event"])} | {seq} |')
    rollback, without_payload = rows
    assert not rollback["projection"] and not rollback["event"] and not rollback["event_sequence"]
    assert without_payload["projection"] and not without_payload["event"] and without_payload["event_sequence"]
elif CHAPTER == "09-projections-storage":
    lines = ["| 时点 | session_inbox | session_message | event 类型与序号 |",
             "|---|---|---|---|"]
    for row in rows:
        inbox = ", ".join(f'{x["delivery"]} @{x["enqueued_seq"]}' for x in row["session_inbox"]) or "空"
        messages = ", ".join(f'{x["type"]} @{x["seq"]}' for x in row["session_message"]) or "空"
        events = ", ".join(f'{x["type"]} @{x["seq"]}' for x in row["event"]) or "空"
        lines.append(f'| {row["phase"]} | {inbox} | {messages} | {events} |')
    pending, finished = rows
    assert pending["session_inbox"][0]["id"] == finished["session_message"][0]["id"]
    assert not finished["session_inbox"]
    print("ID 核对通过 · 消费 Inbox 行后，用同一 ID 插入 user 消息")
else:
    lines = ["| 时点 | 子 Session claim / attempts | Job 状态 | 父 Inbox 条数 | drain 调用 |",
             "|---|---|---|---:|---|"]
    for row in rows:
        sessions = ", ".join(
            f'{"有" if s["claimed_at"] is not None else "无"} / {s["attempts"]}' for s in row["session_v2"]
        )
        jobs = ", ".join(job["status"] for job in row["job"]) or "空"
        drains = ", ".join("子" if name.endswith("_child") else "父" for name in row.get("drains", [])) or "—"
        lines.append(f'| {row["phase"]} | {sessions} | {jobs} | {len(row["parent_inbox"])} | {drains} |')
    before, active, settled, outbox_before, outbox_after, reopened = rows
    assert active["session_v2"][0]["attempts"] == before["session_v2"][0]["attempts"] + 1
    assert any(name.endswith("_child") for name in active["drains"])
    assert outbox_before["job"][0]["status"] == "completed"
    assert outbox_after["drains"] == ["ses_outbox_parent"]
    assert not reopened["job"] and len(reopened["parent_inbox"]) == 2
    print("Bun 退出后重新打开数据库，仍可读到两个父 Inbox 的通知")

display(Markdown("\n".join(lines)))


已重跑原版流程并重新读取 SQLite 行
ID 核对通过 · 消费 Inbox 行后，用同一 ID 插入 user 消息


| 时点 | session_inbox | session_message | event 类型与序号 |
|---|---|---|---|
| admitted | steer @0 | 空 | session.inbox.enqueued.1 @0 |
| run_finished | 空 | user @2, assistant @3 | session.inbox.enqueued.1 @0, session.instructions.updated.2 @1, session.inbox.delivered.1 @2, session.step.started.1 @3, session.step.first-token.1 @4, session.text.started.1 @5, session.text.ended.1 @6, session.step.streamed.1 @7, session.step.ended.1 @8 |